In [78]:
import numpy as np
import pandas as pd

In [79]:
temp_df=pd.read_csv('IMDb Dataset.csv')

In [80]:
df=temp_df.iloc[:10000]

In [81]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [82]:
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [83]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [84]:
df.duplicated().sum()

np.int64(17)

In [85]:
df.drop_duplicates(inplace=True)

In [86]:
df.duplicated().sum()

np.int64(0)

In [87]:
import re
def remove_tags(raw_text):
    cleaned_text=re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [88]:
df['review']=df['review'].apply(remove_tags)

In [89]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
9995,"Fun, entertaining movie about WWII German spy ...",positive
9996,Give me a break. How can anyone say that this ...,negative
9997,This movie is a bad movie. But after watching ...,negative
9998,This is a movie that was probably made to ente...,negative


In [90]:
df['review']=df['review'].apply(lambda x:x.lower())
df

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
...,...,...
9995,"fun, entertaining movie about wwii german spy ...",positive
9996,give me a break. how can anyone say that this ...,negative
9997,this movie is a bad movie. but after watching ...,negative
9998,this is a movie that was probably made to ente...,negative


In [91]:
from nltk.corpus import stopwords
sw_list=stopwords.words('english')

In [92]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in sw_list:
            new_text.append('')
        else:
            new_text.append(word)
    x=new_text[:]
    new_text.clear()
    return " ".join(x) # Join the list of words into a single string and return it

In [93]:
df['review'] = df['review'].apply(remove_stopwords)

In [94]:
x=df.iloc[:,0:1]
y=df['sentiment']

In [95]:
x

,review
0,one reviewers mentioned watching 1 oz e...
1,wonderful little production. filming techniq...
2,thought wonderful way spend time hot s...
3,basically there's family little boy (jake) ...
4,"petter mattei's ""love time money"" visuall..."
...,...
9995,"fun, entertaining movie wwii german spy (juli..."
9996,"give break. anyone say ""good hockey mo..."
9997,movie bad movie. watching endless series...
9998,movie probably made entertain middle sc...


In [96]:
y

0       positive
1       positive
2       positive
3       negative
4       positive
          ...   
9995    positive
9996    negative
9997    negative
9998    negative
9999    positive
Name: sentiment, Length: 9983, dtype: str

In [97]:
from sklearn.preprocessing import LabelEncoder
emcoder=LabelEncoder()
y=emcoder.fit_transform(y)
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [98]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=1)

In [99]:
x_train.shape

(7986, 1)

In [100]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()

In [101]:
x_train_bow=cv.fit_transform(x_train['review']).toarray()
x_test_bow=cv.transform(x_test['review']).toarray()

In [102]:
x_train_bow.shape

(7986, 48282)

In [103]:
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()
gnb.fit(x_train_bow,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [104]:
y_pred=gnb.predict(x_test_bow)

from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
accuracy_score(y_test,y_pred)

0.6324486730095142

In [105]:
confusion_matrix(y_test,y_pred)

array([[717, 235],
       [499, 546]])

In [106]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()

rf.fit(x_train_bow,y_train)
y_pred=rf.predict(x_test_bow)
accuracy_score(y_test,y_pred)

0.8442663995993991

In [107]:
cv=CountVectorizer(max_features=5000)
x_train_bow=cv.fit_transform(x_train['review']).toarray()
x_test_bow=cv.transform(x_test['review']).toarray()
rf=RandomForestClassifier()
rf.fit(x_train_bow,y_train)
y_pred=rf.predict(x_test_bow)
accuracy_score(y_test,y_pred)

0.8392588883324987

In [108]:
cv=CountVectorizer(ngram_range=(1,2), max_features=5000)

In [109]:
x_train_bow = cv.fit_transform(x_train['review']).toarray()
x_test_bow = cv.transform(x_test['review']).toarray()
rf=RandomForestClassifier()
rf.fit(x_train_bow,y_train)
y_pred=rf.predict(x_test_bow)
accuracy_score(y_test,y_pred)

0.8392588883324987

In [110]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [111]:
tfidf=TfidfVectorizer()

In [112]:
x_train_tfidf = tfidf.fit_transform(x_train['review']).toarray()
x_test_tfidf = tfidf.transform(x_test['review']).toarray()

In [114]:
rf=RandomForestClassifier()
rf.fit(x_train_tfidf,y_train)
y_pred=rf.predict(x_test_tfidf)
accuracy_score(y_test,y_pred)

0.8492739108662994